# Are the gaps bigger than chance? (Phase 4: statistics)

Phase 3 found gaps between groups. Here each one gets two checks: **is it bigger than chance
would explain**, and **how big is it** (with a 95% confidence interval). These are still
associations in past orders, not causes. Numbers come from the **analytics** tables built by
`python -m src.pipeline`.

**Contents:** 1. Setup · 2. How sure are we of one rate? · 3. Late vs on time

## 1. Setup
Same connection helper as the first notebook. **SciPy** is Python's standard library for
statistics; we use its tests and intervals instead of writing the formulas ourselves.

In [1]:
import math
import sys
from pathlib import Path

import pandas as pd
from scipy import stats
from sqlalchemy import text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))            # so "from src..." works when run from notebooks/
from src.db import get_engine

engine = get_engine()

def query(sql: str) -> pd.DataFrame:
    """Run one SQL query and return the result as a table."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

## 2. How sure are we of one rate?
The main KPI is the unhappy rate: 1-2 star orders ÷ reviewed orders. Our orders are one sample of
what Olist's customers do, so the rate would wobble a little with a different sample. A **95%
confidence interval** is the range the true rate probably lies in: if we repeated the sampling
many times, 95% of such ranges would contain it.

We use the **Wilson** interval (SciPy's `binomtest(...).proportion_ci`), which stays accurate for
small groups and rates near 0% or 100%. The simple textbook formula, rate ± 1.96 × standard error,
is shown next to it as a check.

In [2]:
def rate_ci(unhappy: int, reviewed: int) -> dict:
    """Unhappy rate with a 95% Wilson interval, plus the simple formula as a check (all in %)."""
    ci = stats.binomtest(unhappy, reviewed).proportion_ci(confidence_level=0.95, method="wilson")
    p = unhappy / reviewed
    se = math.sqrt(p * (1 - p) / reviewed)          # standard error: typical wobble of the rate
    return {"reviewed": reviewed, "unhappy": unhappy, "rate_pct": round(100 * p, 2),
            "wilson_low": round(100 * ci.low, 2), "wilson_high": round(100 * ci.high, 2),
            "simple_low": round(100 * (p - 1.96 * se), 2), "simple_high": round(100 * (p + 1.96 * se), 2)}

kpi = query("""
    SELECT count(*) FILTER (WHERE has_review) AS reviewed, count(*) FILTER (WHERE is_unhappy) AS unhappy
    FROM analytics.fact_order
""").iloc[0]
pd.DataFrame([rate_ci(int(kpi["unhappy"]), int(kpi["reviewed"]))], index=["all reviewed orders"])

,reviewed,unhappy,rate_pct,wilson_low,wilson_high,simple_low,simple_high
all reviewed orders,98673,14533,14.73,14.51,14.95,14.51,14.95


**Group size matters.** The same calculation for a small group from P1: orders 8+ days late whose
customer rated *after* the parcel arrived.

In [3]:
small = query("""
    SELECT count(*) AS reviewed, count(*) FILTER (WHERE is_unhappy) AS unhappy
    FROM analytics.fact_order
    WHERE order_status = 'delivered' AND days_late >= 8 AND has_review
      AND review_answer_timestamp >= order_delivered_customer_date
""").iloc[0]
pd.DataFrame([rate_ci(int(kpi["unhappy"]), int(kpi["reviewed"])),
              rate_ci(int(small["unhappy"]), int(small["reviewed"]))],
             index=["all reviewed orders", "8+ days late, rated after arrival"])

,reviewed,unhappy,rate_pct,wilson_low,wilson_high,simple_low,simple_high
all reviewed orders,98673,14533,14.73,14.51,14.95,14.51,14.95
"8+ days late, rated after arrival",76,23,30.26,21.09,41.33,19.93,40.59


## 3. Late vs on time (P1)
Two groups, two unhappy rates. Two questions:
- **Bigger than chance?** The **chi-square test** asks: if lateness had nothing to do with
  unhappiness, how likely is a gap this big just by luck? That chance is the **p-value**. Below 0.05
  is the usual line for "unlikely to be luck alone".
- **How big?** The gap in percentage points with a 95% interval, and the **risk ratio** (how many
  times more often late orders are unhappy) with SciPy's interval.

In [4]:
def compare(name: str, a_unhappy: int, a_n: int, b_unhappy: int, b_n: int) -> dict:
    """Group A vs group B: gap in points (95% interval), risk ratio (95% interval), chi-square p-value."""
    pa, pb = a_unhappy / a_n, b_unhappy / b_n
    se = math.sqrt(pa * (1 - pa) / a_n + pb * (1 - pb) / b_n)       # wobble of the gap
    table = [[a_unhappy, a_n - a_unhappy], [b_unhappy, b_n - b_unhappy]]
    p_value = stats.chi2_contingency(table).pvalue
    rr = stats.contingency.relative_risk(a_unhappy, a_n, b_unhappy, b_n)
    rr_ci = rr.confidence_interval(confidence_level=0.95)
    return {"comparison": name, "n_a": a_n, "rate_a_pct": round(100 * pa, 2),
            "n_b": b_n, "rate_b_pct": round(100 * pb, 2),
            "gap_points": round(100 * (pa - pb), 2),
            "gap_low": round(100 * (pa - pb - 1.96 * se), 2), "gap_high": round(100 * (pa - pb + 1.96 * se), 2),
            "risk_ratio": round(rr.relative_risk, 2), "rr_low": round(rr_ci.low, 2), "rr_high": round(rr_ci.high, 2),
            "p_value": f"{p_value:.1e}"}

In [5]:
# Your Step 2 query, plus the same counts for customers who rated AFTER the parcel arrived
p1 = query("""
    SELECT CASE WHEN days_late > 0 THEN 'late' ELSE 'on time' END AS promise,
           count(*)                           AS reviewed,
           count(*) FILTER (WHERE is_unhappy) AS unhappy,
           count(*) FILTER (WHERE review_answer_timestamp >= order_delivered_customer_date) AS reviewed_after,
           count(*) FILTER (WHERE is_unhappy AND review_answer_timestamp >= order_delivered_customer_date)
               AS unhappy_after
    FROM analytics.fact_order
    WHERE order_status = 'delivered' AND days_late IS NOT NULL AND has_review
    GROUP BY promise
""").set_index("promise")
late, on_time = p1.loc["late"], p1.loc["on time"]

pd.DataFrame([
    compare("late vs on time (all reviewed)",
            int(late.unhappy), int(late.reviewed), int(on_time.unhappy), int(on_time.reviewed)),
    compare("late vs on time (rated after arrival)",
            int(late.unhappy_after), int(late.reviewed_after), int(on_time.unhappy_after), int(on_time.reviewed_after)),
]).set_index("comparison").T

comparison,late vs on time (all reviewed),late vs on time (rated after arrival)
n_a,6381,1907
rate_a_pct,62.47,19.45
n_b,89443,89242
rate_b_pct,9.31,9.28
gap_points,53.16,10.18
gap_low,51.96,8.39
gap_high,54.36,11.96
risk_ratio,6.71,2.1
rr_low,6.53,1.91
rr_high,6.9,2.3


The gap is far bigger than chance in both versions. Counting only customers who rated after the
parcel arrived, the gap shrinks a lot but stays clear. The groups still differ in other ways
(routes, products, sellers), so this is a link, not proof that lateness is the reason.